# Musanga V4 — NB03 V4.002 | Integrated RGB+DSM U-Net training and paired spatial validation

**One Colab workflow:** read frozen V4 field78 sources and fold manifest; load DSM pairs; compute three local surface-geometry channels; train one six-channel ResNet18 U-Net for all three outer folds (masked BCE, blocked exhaustive-only inner selection); compare with frozen RGB V4.007 using identical native-pixel DSM-valid supports and **50% Hann-blended inference** on all held-out field plots and 34 circles. Results: `Final/Outputs/v4/03_rgb_dsm_field78_v4_002/paired_evaluation`.

**Preconditions:** NB00/01 reference manifests and V4.007 RGB checkpoints must exist. Run on Colab GPU. This is computationally substantial. No V3_1, field inputs, RGB checkpoints or NB02 outputs are overwritten. Check local DSM co-registration visually in addition to the automated validity tests: DSM is *SfM surface*, **not CHM**. The model uses RGB+prominence+roughness+convexity; extra ablations/MS/LAS can follow if DSM helps.


## 00 — Environment and frozen reference

In [ ]:
# 00 — Environment, frozen inputs and run settings
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-learn torch matplotlib segmentation-models-pytorch
from pathlib import Path
import json, math, random, copy, time, warnings, os
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
ROOT=DRIVE/'Colab Notebooks/Musanga/Final'
assert ROOT.is_dir(), ROOT
DATA=ROOT/'Data'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF_DIR/'01_field78_reference_contract.json').read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2',CONTRACT['schema']
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
assert NB00['n_field_plots']==78
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096,(len(SPLITS),'Expected 3032 unique windows × 3 folds; changed source audit?')
K=['cv_fold','mosaic','row','col','width','height']
assert not SPLITS.duplicated(K).any()
assert SPLITS.groupby('cv_fold').size().eq(3032).all()
assert set(SPLITS.split).issubset({'train','val','excluded_spatial_guard','excluded_no_validation_pixels'})
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PLOT_FOLDS=pd.read_csv(CONTRACT['folds_csv'])
assert len(PLOTS)==len(PLOT_FOLDS)==78 and PLOT_FOLDS.source_id.is_unique
PLOTS=PLOTS.merge(PLOT_FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',right_on='source_id',validate='one_to_one')
CIRCLE_FOLDS=pd.read_csv(REF_DIR/'01_field78_circle_folds.csv')
# Adjustable, deterministic experiment settings
SMOKE_TEST=False           # FIXED FULL RUN ONLY; no smoke option
RUN_FOLDS=[0,1,2]         # FIXED all three spatial folds
SEED=20261113
CHIP=256                   # 256 native pixels; no spatial downsampling
BATCH_SIZE=8
EPOCHS=25
STEPS_PER_EPOCH=100
EVAL_BATCH_SIZE=8  # batched exhaustive held-out inference, mosaic handle opened once
LEARNING_RATE=2e-4
PATIENCE=4
POSITIVE_SAMPLE_BOOST=2.5
MAX_VAL_WINDOWS_PER_SITE=35  # early-stopping cache only; FINAL evaluation is uncapped
MAX_TRAIN_WINDOWS_PER_SITE=180
NUM_WORKERS=0              # Google Drive + GDAL: safest in Colab
OUT=ROOT/'Outputs/v4/03_rgb_dsm_field78_v4_002'
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda', 'V3_1 used GPU; select Colab GPU runtime before training'
print('Device:',DEVICE,'| NESTED FULL SPATIAL CV')
print('Final manifest:',CONTRACT['cv_manifest'])
print('Observed window splits:');display(SPLITS.groupby(['cv_fold','site','split']).size().unstack(fill_value=0))

# Canonical immutable numbered experiment; no shared output with earlier experiments.
VERSION='V4.007'
assert SMOKE_TEST is False and RUN_FOLDS==[0,1,2]
assert EPOCHS==25 and STEPS_PER_EPOCH==100
assert MAX_TRAIN_WINDOWS_PER_SITE==180 and MAX_VAL_WINDOWS_PER_SITE==35
assert OUT.name=='03_rgb_dsm_field78_v4_002',OUT
CHECKPOINTS={fold:OUT/f'NB03_V4_002_fold{fold}_best.pt' for fold in RUN_FOLDS}
print('DSM EXPERIMENT: V4.002 | frozen RGB reference:',VERSION,'\nOUTPUT:',OUT)
print('FOLD CHECKPOINTS:',{f:str(p) for f,p in CHECKPOINTS.items()})

N_BINS=512 # used by secondary complete-circle metrics if plot evaluation is skipped

INNER_BLOCK_PX=2048
INNER_GUARD_PX=256
INNER_VAL_FRACTION=.20
SOURCE_CODES={0:'unknown',1:'partial_crown',2:'exhaustive_circle',3:'exhaustive_plot'}

# V4.007 reuses ONLY V4.003 source-labelled chip cache bytes when exact
# manifest/input fingerprint matches; no model weights are transferred.
READ_ONLY_CACHE_V003=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_003_full/chip_cache'
INNER_SELECTION_SOURCES=(2,3)   # exhaustive circles and plots ONLY
TRAINING_SOURCES=(1,2,3)         # individual positive/negative crowns training ONLY
assert VERSION=='V4.007' and OUT.name=='03_rgb_dsm_field78_v4_002'

import gc,resource
def memory_snapshot(tag):
    print('RAM diagnostic',tag,'max_RSS_GB',round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024**2,2),flush=True)

BASELINE_DIR=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_007_full'
BASELINE_CHECKPOINTS={f:BASELINE_DIR/f'NB02_V4_007_fold{f}_best.pt' for f in RUN_FOLDS}
for p in BASELINE_CHECKPOINTS.values(): assert p.is_file(),p


## 01 — Pixel references, geometry and strict label masks (NB02 exact semantics)

In [ ]:
# 01 — Geometry references and fast spatial indexes
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CIRCLE_ROI=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CIRCLE_POS=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(CROWNS)==3324 and len(CIRCLE_ROI)==34
assert 'Class_ID' in CROWNS
def polygons(g):
    if g is None or g.is_empty:return []
    if g.geom_type=='Polygon':return [g]
    if g.geom_type=='MultiPolygon':return list(g.geoms)
    return [a for v in getattr(g,'geoms',[]) for a in polygons(v)]
def fix(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    q=unary_union(polygons(make_valid(g)))
    if q.is_empty or not q.is_valid:return None
    if abs(q.area-g.area)/max(q.area,g.area,1e-12)>.05:return None
    return q
CROWNS.geometry=[fix(g) for g in CROWNS.geometry]
assert CROWNS.geometry.notna().all(),'Unrepaired plot crowns; see NB00 QA'
CIRCLE_ROI.geometry=[fix(g) for g in CIRCLE_ROI.geometry]
CIRCLE_POS.geometry=[fix(g) for g in CIRCLE_POS.geometry]
assert CIRCLE_ROI.geometry.notna().all()
CIRCLE_ROI['circle_id']=np.arange(len(CIRCLE_ROI))
CIRCLE_ROI=CIRCLE_ROI.merge(CIRCLE_FOLDS[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWN_POS=list(CROWNS.loc[pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CROWN_NEG=list(CROWNS.loc[~pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CIRCLE_POS_GEOMS=[g for g in CIRCLE_POS.geometry if g is not None and not g.is_empty]
# Partial site references from old V3_1 audit, same positive=1 / other known class negative.
PARTIAL={}
for site,path in NB00['partial_site_crowns'].items():
    d=gpd.read_file(path).to_crs('EPSG:32635')
    d.geometry=[fix(g) for g in d.geometry]
    d=d[d.geometry.notna() & pd.to_numeric(d.Class_ID,errors='coerce').notna()]
    PARTIAL[site]=d
P_GEOMS=list(PLOTS.geometry)
C_GEOMS=list(CIRCLE_ROI.geometry)
P_T=STRtree(P_GEOMS); C_T=STRtree(C_GEOMS)
CP_T=STRtree(CROWN_POS);CN_T=STRtree(CROWN_NEG)
KPOS_T=STRtree(CIRCLE_POS_GEOMS)
PARTIAL_T={}
for site,d in PARTIAL.items():
    pos=list(d.loc[pd.to_numeric(d.Class_ID).eq(1)].geometry)
    neg=list(d.loc[~pd.to_numeric(d.Class_ID).eq(1)].geometry)
    PARTIAL_T[site]=(pos,neg,STRtree(pos) if pos else None,
                     STRtree(neg) if neg else None)
RGB={}
for item in sorted(SPLITS.mosaic.unique()):
    q=DATA/'Ortho_RGB'/item
    assert q.exists(),q
    with rasterio.open(q) as ds:
        assert ds.crs.to_epsg()==32635,(item,ds.crs)
        assert ds.count>=3,(item,ds.count)
        RGB[item]=dict(path=str(q),transform=ds.transform,site=next(
            iter(SPLITS.loc[SPLITS.mosaic==item,'site'].unique())))
# NOTE: ROI.shp is the authoritative six-mosaic modelling ROI (not the old
# circle-only support). The validation/labelled field plot is also required.
ROIS=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
ROI_GEOMS=list(ROIS.geometry)
ROI_T=STRtree(ROI_GEOMS)
print('Loaded',len(PLOTS),'plots,',len(CIRCLE_ROI),'circles,',
      len(CROWN_POS),'Musanga plot crowns and',len(CIRCLE_POS_GEOMS),'circle crowns')


In [ ]:
# 02 — Rasterize strict plot > circle > partial supervision on demand
# Geometry-level spatial queries restrict operations to the local 256px footprint.
def geom_hits(tree,geoms,ext):
    if tree is None:return []
    return [geoms[int(i)] for i in tree.query(ext,predicate='intersects')]
_ACTIVE_RASTER=None
_ACTIVE_NAME=None
def rgb_window(raster_path,win):
    from contextlib import nullcontext
    name=Path(raster_path).name
    ds_context=nullcontext(_ACTIVE_RASTER) if (_ACTIVE_RASTER is not None and _ACTIVE_NAME==name) else rasterio.open(raster_path)
    with ds_context as ds:
        a=ds.read([1,2,3],window=win,masked=True)
        valid=ds.dataset_mask(window=win)>0
        valid &= ~np.ma.getmaskarray(a).any(axis=0)
        x=np.asarray(a.filled(0),dtype=np.float32)
        # RGB products expected standard 8-bit; prevent silent incorrect scaling.
        if np.issubdtype(np.dtype(ds.dtypes[0]),np.integer) and np.dtype(ds.dtypes[0]).itemsize==1:
            x=x/255.
        else:
            raise RuntimeError(f'Non-uint8 RGB {ds.dtypes}: {raster_path}; define radiometric scaling first')
    return x,np.asarray(valid,bool)
def supervision(mosaic,win,fold,role,with_source=False):
    site=RGB[mosaic]['site'];t=RGB[mosaic]['transform']
    h,w=int(win.height),int(win.width)
    affine=rasterio.windows.transform(win,t)
    footprint=box(*rasterio.windows.bounds(win,t))
    def burn(gs):
        return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
            out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros((h,w),bool)
    roi=burn(geom_hits(ROI_T,ROI_GEOMS,footprint))
    pids=[int(i) for i in P_T.query(footprint,predicate='intersects')
          if PLOTS.iloc[int(i)].site_key==site and
          (role=='train' and PLOTS.iloc[int(i)].cv_fold!=fold or
           role=='val' and PLOTS.iloc[int(i)].cv_fold==fold)]
    plot=burn([P_GEOMS[i] for i in pids])
    # Every other fold's field plot is out of scope for val; all exhaustive
    # plot geometries (not only selected) override circle labels.
    allplot=burn([P_GEOMS[int(i)] for i in P_T.query(footprint,predicate='intersects')
                  if PLOTS.iloc[int(i)].site_key==site])
    cids=[int(i) for i in C_T.query(footprint,predicate='intersects')
          if CIRCLE_ROI.iloc[int(i)].site_key==site and
          not CIRCLE_ROI.iloc[int(i)].cross_fold_plot_intersection and
          (role=='train' and CIRCLE_ROI.iloc[int(i)].cv_fold!=fold)]
    circle=burn([C_GEOMS[i] for i in cids])&~allplot if role=='train' else np.zeros((h,w),bool)
    # Partial crowns must NEVER relabel any exhaustive circle, including
    # an outer-held-out circle. Train circles retain exclusive authority.
    allcircle=burn([C_GEOMS[int(i)] for i in C_T.query(footprint,predicate='intersects')
        if CIRCLE_ROI.iloc[int(i)].site_key==site])
    posplot=burn(geom_hits(CP_T,CROWN_POS,footprint))&plot
    negplot=burn(geom_hits(CN_T,CROWN_NEG,footprint))&plot
    poscircle=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint))&circle
    part_pos=np.zeros((h,w),bool);part_neg=np.zeros((h,w),bool)
    if role=='train':
        positives,negatives,tp,tn=PARTIAL_T[site]
        part_pos=burn(geom_hits(tp,positives,footprint))&~allplot&~allcircle
        part_neg=burn(geom_hits(tn,negatives,footprint))&~allplot&~allcircle
    source=np.zeros((h,w),dtype=np.uint8)
    source[part_pos|part_neg]=1
    source[circle]=2
    source[plot]=3
    target=(posplot|poscircle|part_pos)
    conflicting=(posplot&negplot)|(part_pos&part_neg)
    weight=roi&(plot|circle|part_pos|part_neg)&~conflicting
    source[~weight.astype(bool)]=0
    assert not np.any((source==0)&(weight>0))
    return (target.astype(np.float32),weight.astype(np.float32),plot.astype(bool),source) if with_source else (target.astype(np.float32),weight.astype(np.float32),plot.astype(bool))
def get_chip(mosaic,win,fold,role):
    x,rgbvalid=rgb_window(RGB[mosaic]['path'],win)
    y,w,plot,source=supervision(mosaic,win,fold,role,with_source=True)
    w*=rgbvalid.astype(np.float32)
    source[~rgbvalid]=0
    return x,y,w,plot,source
print('Prepared lazy RGB + spatially strict masks. No imagery loaded globally.')


## 02 — DSM pairing and spatially windowed surface features

In [ ]:
# 02A — Load NB00 DSM audit candidates; no silent alternate DSM mapping
NB00_AUDIT=ROOT/'Outputs/v4/00_input_audit/00_rgb_dsm_pair_audit.csv'
assert NB00_AUDIT.is_file(),NB00_AUDIT
PAIRS=pd.read_csv(NB00_AUDIT)
assert len(PAIRS)==6 and PAIRS.rgb.is_unique
PROMINENCE_RADIUS_M=6.
ROUGHNESS_RADIUS_M=3.
CONVEXITY_RADIUS_M=2.
MIN_JOINT_VALID_FRACTION=.95

# 01 — Conservative pair registry. Approve only after registration QA.
def mosaic_key(name):
    v=str(name).casefold().replace(' ','_')
    if 'baego' in v:return 'Baego'
    if 'mos' in v:
        for k in ('1_2','3_4','5_6'):
            if k in v:return 'MOS_'+k
        return 'MOS_unknown'
    if 'yangambi' in v or 'ybjoseph' in v or 'ybforgeo' in v or 'carlier' in v:return 'Yangambi'
    if 'yoko' in v:return 'Yoko'
    return 'unknown'

REGISTRY={}
for rec in PAIRS.itertuples(index=False):
    rgb=Path(rec.rgb_path);dsm=Path(rec.dsm_path)
    assert rgb.is_file() and dsm.is_file(),(rgb,dsm)
    assert mosaic_key(rgb.name)==mosaic_key(dsm.name),(rgb.name,dsm.name)
    assert float(rec.roi_overlap_fraction)>0,(
        'DSM missing completely from ROI; cannot match',
        rgb.name,rec.roi_overlap_fraction)
    with rasterio.open(rgb) as a,rasterio.open(dsm) as b:
        assert a.crs is not None and b.crs is not None
        assert a.count>=3 and b.count>=1
        assert a.crs.to_epsg()==32635,(rgb,a.crs)
        px=(abs(a.transform.a),abs(a.transform.e))
        assert 0<min(px)<100,(rgb,px)
        REGISTRY[rgb.name]=dict(rgb=str(rgb),dsm=str(dsm),
            site=mosaic_key(rgb.name),px=px,transform=a.transform,
            shape=(a.height,a.width),dsm_count=b.count,
            rgb_crs=str(a.crs),dsm_crs=str(b.crs))
display(pd.DataFrame([dict(rgb=k,dsm=Path(v['dsm']).name,
      rgb_resolution_m=v['px'][0],dsm_bands=v['dsm_count'])
      for k,v in REGISTRY.items()]))
(OUT/'03_pair_registry_provisional.json').write_text(json.dumps(
    {k:{z:(list(q) if isinstance(q,tuple) else str(q) if z=='transform' else q)
        for z,q in v.items()} for k,v in REGISTRY.items()},indent=2))
print('WARNING: provisional pairs. Bbox overlap alone does NOT certify registration.')


In [ ]:
# 02 — Windowed, halo-safe structural feature extraction
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter
def _mean_valid(a,valid,size):
    f=valid.astype(np.float32)
    num=uniform_filter(np.where(valid,a,0.).astype(np.float32),size=size,mode='nearest')
    den=uniform_filter(f,size=size,mode='nearest')
    return num/np.maximum(den,1e-6),den

def _odd_pixels(radius_m,pixel_m):
    r=max(1,int(np.ceil(radius_m/pixel_m)))
    return 2*r+1

def feature_tile(rgb_path,dsm_path,win):
    """Return 6xHxW [RGB3, prominence, roughness, convexity], joint valid HxW.
    Requires actual pixel registration to be separately approved before training.
    """
    with ExitStack() as stack:
        rgb=stack.enter_context(rasterio.open(rgb_path))
        dsm=stack.enter_context(rasterio.open(dsm_path))
        assert rgb.crs is not None and dsm.crs is not None
        h,w=int(win.height),int(win.width)
        assert h>0 and w>0 and int(win.row_off)>=0 and int(win.col_off)>=0
        assert win.row_off+h<=rgb.height and win.col_off+w<=rgb.width
        step=min(abs(rgb.transform.a),abs(rgb.transform.e))
        max_radius=max(PROMINENCE_RADIUS_M,ROUGHNESS_RADIUS_M,
                       CONVEXITY_RADIUS_M)
        halo=int(np.ceil(2.0*max_radius/step))+4
        r0=max(0,int(win.row_off)-halo);c0=max(0,int(win.col_off)-halo)
        r1=min(rgb.height,int(win.row_off)+h+halo)
        c1=min(rgb.width,int(win.col_off)+w+halo)
        expanded=Window(c0,r0,c1-c0,r1-r0)
        tr=rgb.window_transform(expanded)
        rgb_raw=rgb.read([1,2,3],window=expanded,masked=True)
        if not all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3]):
            raise ValueError(f'RGB is not uint8: {rgb_path}: {rgb.dtypes}')
        rgb_valid=(rgb.dataset_mask(window=expanded)>0)&(
            ~np.ma.getmaskarray(rgb_raw).any(axis=0))
        rgb_data=np.asarray(rgb_raw.filled(0),dtype=np.float32)/255.
        vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,transform=tr,
            width=int(expanded.width),height=int(expanded.height),
            resampling=Resampling.bilinear,src_nodata=dsm.nodata,
            nodata=np.nan))
        arr=vrt.read(1,masked=True)
        z=np.asarray(arr.data,dtype=np.float32).copy()
        dsm_valid=(~np.ma.getmaskarray(arr))&np.isfinite(z)
        valid=rgb_valid&dsm_valid
        z0=np.where(dsm_valid,z,0.).astype(np.float32)
        prominence_size=_odd_pixels(PROMINENCE_RADIUS_M,step)
        rough_size=_odd_pixels(ROUGHNESS_RADIUS_M,step)
        convex_size=_odd_pixels(CONVEXITY_RADIUS_M,step)
        base,base_support=_mean_valid(z0,dsm_valid,prominence_size)
        local,rough_support=_mean_valid(z0,dsm_valid,rough_size)
        local2,_=_mean_valid(z0*z0,dsm_valid,rough_size)
        smooth,conv_support=_mean_valid(z0,dsm_valid,convex_size)
        prominence=z0-base
        roughness=np.sqrt(np.maximum(0.,local2-local**2))
        convexity=z0-smooth
        # Exclude filters with inadequate valid support near DSM boundaries.
        valid &= (base_support>=.95)&(rough_support>=.95)&(conv_support>=.95)
        y=int(win.row_off)-r0;x=int(win.col_off)-c0
        sl=np.s_[y:y+h,x:x+w]
        x6=np.concatenate([rgb_data[:,sl[0],sl[1]],
            prominence[sl][None],roughness[sl][None],
            convexity[sl][None]],axis=0).astype(np.float32)
        joint=valid[sl].copy()
        x6[:,~joint]=0.
        assert x6.shape==(6,h,w) and joint.shape==(h,w)
        assert np.isfinite(x6).all()
        return x6,joint

FEATURE_NAMES=('R','G','B','local_prominence_m',
               'surface_roughness_m','local_convexity_m')
print('Available features:',FEATURE_NAMES)


## 03 — Source-labelled cache and inner spatial CV, inherited from NB02 V4.007

In [ ]:
# 03 — V3_1-style resumable cache, file provenance, persistent raster handles
from contextlib import contextmanager
import hashlib, shutil
from collections import defaultdict
from rasterio.windows import Window
CACHE_DIR=OUT/'chip_cache';CACHE_DIR.mkdir(parents=True,exist_ok=True)
USE_LOCAL_STAGE=False  # Same optional one-mosaic SSD staging as V3_1; disabled by default
STAGE_DIR=Path('/content/musanga_v4_stage');STAGE_DIR.mkdir(parents=True,exist_ok=True)
@contextmanager
def staged_mosaic(name):
    global _ACTIVE_RASTER,_ACTIVE_NAME
    src=Path(RGB[name]['path']);selected=src
    local=STAGE_DIR/src.name
    if USE_LOCAL_STAGE and shutil.disk_usage('/content').free>src.stat().st_size+2*1024**3:
        if not local.exists() or local.stat().st_size!=src.stat().st_size:
            tmp=local.with_name(local.name+'.part')
            with src.open('rb') as a,tmp.open('wb') as b:
                shutil.copyfileobj(a,b,16*1024*1024)
            tmp.replace(local)
        selected=local
    assert _ACTIVE_RASTER is None
    try:
        with rasterio.open(selected) as ds:
            assert ds.transform==RGB[name]['transform'] and ds.count>=3
            _ACTIVE_RASTER=ds;_ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None;_ACTIVE_NAME=None
        if selected==local:local.unlink(missing_ok=True)
def cache_signature(fold,role):
    dependencies=[Path(CONTRACT['cv_manifest']),Path(CONTRACT['input_manifest']),
        Path(NB00['exhaustive_plot_crowns']),Path(NB00['plot_file']),
        Path(NB00['circle_positive_crowns']),Path(NB00['circle_complete_roi'])]
    dependencies += [Path(p) for p in NB00['partial_site_crowns'].values()]
    source_state=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in dependencies]
    info=dict(version='v4_003_source_nested_v1',fold=int(fold),role=role,
        seed=SEED,chip=CHIP,smoke=False,src=source_state,
        max_train=MAX_TRAIN_WINDOWS_PER_SITE,max_val=MAX_VAL_WINDOWS_PER_SITE)
    return hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest()
def pack(path,items,signature):
    tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,key=np.array(signature),
        rgb=np.stack([r['rgb'] for r in items]).astype(np.uint8),
        y=np.stack([r['y'] for r in items]).astype(np.uint8),
        w=np.stack([r['w'] for r in items]).astype(np.uint8),
        source=np.stack([r['source'] for r in items]).astype(np.uint8),
        site=np.array([r['site'] for r in items],dtype='U20'),
        mosaic=np.array([r['mosaic'] for r in items],dtype='U100'),
        row=np.array([r['row'] for r in items],dtype=np.int32),
        col=np.array([r['col'] for r in items],dtype=np.int32))
    tmp.replace(path)
def unpack(path,signature):
    """Read each compressed NPZ member EXACTLY ONCE.

    Repeated z['rgb'][i] access decompresses the full image tensor per
    chip. For ~650 256px chips this can trigger an abrupt Colab RAM kill.
    """
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature,('Cache fingerprint mismatch',path)
        rgb=z['rgb']
        y=z['y'];w=z['w'];source=z['source']
        site=z['site'];mosaic=z['mosaic']
        row=z['row'];col=z['col']
    n=len(rgb)
    assert y.shape==w.shape==source.shape==rgb.shape[:3]
    assert len(site)==len(mosaic)==len(row)==len(col)==n
    return [dict(rgb=rgb[i],y=y[i],w=w[i],source=source[i],
                 site=str(site[i]),mosaic=str(mosaic[i]),
                 row=int(row[i]),col=int(col[i])) for i in range(n)]

def source_window_rows(fold,role):
    q=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split==role)].copy()
    assert len(q)>0,(fold,role)
    cap=MAX_TRAIN_WINDOWS_PER_SITE if role=='train' else MAX_VAL_WINDOWS_PER_SITE
    q['likely_positive']=q.apply(lambda r:bool(
        CP_T.query(box(*rasterio.windows.bounds(
            Window(int(r.col),int(r.row),int(r.width),int(r.height)),
            RGB[r.mosaic]['transform'])),predicate='intersects').size),axis=1)
    selected=[]
    for site,g in q.groupby('site',sort=True):
        # Approximate stratification by positive intersection before any image reads.
        high=g[g.likely_positive];low=g[~g.likely_positive]
        if role=='train' and len(high) and len(low):
            nh=min(len(high),int(cap*.65))
            chosen=pd.concat([high.sample(n=nh,random_state=SEED+fold),
                 low.sample(n=min(len(low),cap-nh),random_state=SEED+fold)])
        else:
            chosen=g.sample(n=min(cap,len(g)),random_state=SEED+fold)
        selected.append(chosen)
    return pd.concat(selected,ignore_index=True)
def make_cache(fold,role):
    sig=cache_signature(fold,role)
    cache=CACHE_DIR/f'field78_{role}_fold{fold}_{sig[:16]}.npz'
    # Provenance-safe cache reuse: same source labels and raw inputs as V4.003.
    # Read only; never write, rename or delete the previous experiment's cache.
    old=READ_ONLY_CACHE_V003/cache.name
    if not cache.exists() and old.is_file():
        try:
            items=unpack(old,sig)
            assert all('source' in item for item in items)
            print('READ-ONLY V4.003 LABELLED CACHE REUSE',role,'fold',fold,
                  'chips',len(items),flush=True)
            return items
        except Exception as exc:
            print('Legacy cache fingerprint mismatch, rebuild in V4.007:',exc)
    if cache.exists():
        try:
            items=unpack(cache,sig)
            print('CACHE HIT',role,'fold',fold,'n',len(items),flush=True)
            return items
        except Exception as exc:
            print('Invalid cache:',exc)
            cache.unlink(missing_ok=True)
    selected=source_window_rows(fold,role)
    print('PREPARE',role,'fold',fold,
        selected.groupby('site').size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{role}_fold{fold}_{sig[:16]}'
    parts_dir.mkdir(exist_ok=True)
    items=[]
    t0=time.perf_counter()
    for mosaic,g in selected.groupby('mosaic',sort=True):
        with staged_mosaic(mosaic):
            for k in range(0,len(g),50):
                part=parts_dir/f'{mosaic}_{k:04d}.npz'
                if part.exists():
                    try:
                        existing=unpack(part,sig)
                        items.extend(existing);continue
                    except Exception:part.unlink(missing_ok=True)
                batch=[]
                for r in list(g.iloc[k:k+50].itertuples(index=False)):
                    # Same deterministic centre crop as V3_1; do not read 512px
                    # merely to throw away 3/4 of the source pixel data.
                    if r.width<CHIP or r.height<CHIP:continue
                    row=int(r.row)+(int(r.height)-CHIP)//2
                    col=int(r.col)+(int(r.width)-CHIP)//2
                    win=Window(col,row,CHIP,CHIP)
                    x,y,w,_,source=get_chip(mosaic,win,fold,role)
                    if w.sum()<50:continue
                    batch.append(dict(rgb=np.rint(np.clip(x.transpose(1,2,0),0,1)*255).astype('uint8'),
                        y=y.astype('uint8'),w=w.astype('uint8'),source=source.astype('uint8'),
                        site=RGB[mosaic]['site'],mosaic=mosaic,row=row,col=col))
                if batch:
                    pack(part,batch,sig)
                    items.extend(batch)
                print('Cached',role,'fold',fold,mosaic,
                    min(k+50,len(g)),'/',len(g),'usable',len(items),
                    'elapsed_s',round(time.perf_counter()-t0,1),flush=True)
    assert len(items)>0,(fold,role,'No valid RGB+reference chips')
    pack(cache,items,sig)
    print('CACHE SAVED',role,'fold',fold,'chips',len(items),
        'seconds',round(time.perf_counter()-t0,1),flush=True)
    return items


In [ ]:
# V4.007: disjoint blocked INNER checkpoint-selection data, from outer TRAIN only.
# Partial labels are allowed for training only; 2=exhaustive circle, 3=plot.
from scipy.spatial import cKDTree
import hashlib

def _exhaustive_stats(items,indices):
    p=n=0
    for j in indices:
        q=items[int(j)]
        mask=(q['source']>=2)&q['w'].astype(bool)
        count=int(mask.sum())
        p+=int((mask&q['y'].astype(bool)).sum());n+=count-int((mask&q['y'].astype(bool)).sum())
    return p,n

def _chip_units(item):
    mosaic=item['mosaic'];win=Window(item['col'],item['row'],CHIP,CHIP)
    ext=box(*rasterio.windows.bounds(win,RGB[mosaic]['transform']))
    site=item['site']
    plots={int(i) for i in P_T.query(ext,predicate='intersects')
           if PLOTS.iloc[int(i)].site_key==site}
    circles={int(i) for i in C_T.query(ext,predicate='intersects')
             if CIRCLE_ROI.iloc[int(i)].site_key==site}
    return plots,circles

def inner_spatial_split(items,fold):
    df=pd.DataFrame([dict(i=i,site=r['site'],mosaic=r['mosaic'],
        row=int(r['row']),col=int(r['col']),
        block_r=int(r['row'])//INNER_BLOCK_PX,
        block_c=int(r['col'])//INNER_BLOCK_PX) for i,r in enumerate(items)])
    df['block_key']=df.apply(lambda r:
         f"{r.mosaic}|{int(r.block_r)}|{int(r.block_c)}",axis=1)
    val_keys=set()
    for site,d in df.groupby('site'):
        blocks={k:v.i.to_numpy() for k,v in d.groupby('block_key')}
        assert len(blocks)>=3,('Insufficient independent spatial blocks',fold,site)
        eligible=[k for k,ids in blocks.items() if _exhaustive_stats(items,ids)[0]>0]
        assert eligible,('No positive exhaustive block for inner model selection',fold,site)
        order=sorted(blocks,key=lambda k:hashlib.sha256(
            f'{SEED}|{fold}|{site}|{k}'.encode()).hexdigest())
        # Prefer a block with exhaustive POSITIVES. Preserve positives in training too.
        seed_candidates=[k for k in order if k in eligible]
        seed=next((k for k in seed_candidates
            if len(blocks[k])/len(d)<=.35 and
            _exhaustive_stats(items,d.loc[~d.block_key.eq(k),'i'])[0]>0),None)
        assert seed,('No positive block allows positive exhaustive training',fold,site)
        chosen=[seed]
        target=max(1,int(round(len(d)*INNER_VAL_FRACTION)))
        for key in order:
            if key in chosen:continue
            if sum(len(blocks[k]) for k in chosen)>=target:break
            if (sum(len(blocks[k]) for k in chosen)+len(blocks[key]))/len(d)>.35:continue
            chosen.append(key)
        fraction=sum(len(blocks[k]) for k in chosen)/len(d)
        assert .08<=fraction<=.35,('Unbalanced inner validation proportion',fold,site,fraction)
        val_keys.update(chosen)
    val_df=df[df.block_key.isin(val_keys)].copy()
    train_df=df[~df.block_key.isin(val_keys)].copy()
    # No individual exhaustive plot/circle may contribute to both inner roles.
    # This guard is stricter than block separation alone.
    val_plots=set();val_circles=set()
    for j in val_df.i:
        pp,cc=_chip_units(items[int(j)])
        val_plots.update(pp);val_circles.update(cc)
    keep=[]
    for row in train_df.itertuples():
        item=items[int(row.i)]
        pp,cc=_chip_units(item)
        if pp&val_plots or cc&val_circles:
            keep.append(False);continue
        v=val_df.loc[val_df.mosaic.eq(row.mosaic)]
        if v.empty:keep.append(True);continue
        dr=v.row.to_numpy(float)-row.row
        dc=v.col.to_numpy(float)-row.col
        keep.append(bool(np.all(dr*dr+dc*dc >= (CHIP+INNER_GUARD_PX)**2)))
    train_df=train_df.loc[np.asarray(keep,bool)].copy()
    train=[items[int(i)] for i in train_df.i]
    val=[]
    for j in val_df.i:
        raw=items[int(j)]
        item=dict(raw)
        item['w']=(raw['w'].astype(bool)&np.isin(raw['source'],INNER_SELECTION_SOURCES)).astype('uint8')
        item['source']=np.where(item['w']>0,raw['source'],0).astype('uint8')
        if item['w'].any():val.append(item)
    assert len(train)>=100 and len(val)>=10,(fold,len(train),len(val))
    assert set(train_df.block_key).isdisjoint(set(val_df.block_key))
    assert not (set(train_df.i)&set(val_df.i))
    assert set(r['site'] for r in train)==set(r['site'] for r in items)
    assert set(r['site'] for r in val)==set(r['site'] for r in items)
    for site in sorted(set(r['site'] for r in items)):
        a=[i for i in train if i['site']==site];b=[i for i in val if i['site']==site]
        trpos,trneg=_exhaustive_stats(a,range(len(a)))
        vp,vn=_exhaustive_stats(b,range(len(b)))
        assert min(trpos,trneg,vp,vn)>0,(
           'Both inner training and selection must contain exhaustive positives/negatives',
           fold,site,trpos,trneg,vp,vn)
        print(f'Fold {fold} {site}: exhaustive inner train {trpos:,}+/{trneg:,}-, '
              f'selection {vp:,}+/{vn:,}-')
    # Audit: selection has no partial positive or negative crown pixels.
    assert all(not np.any((i['source']==1)&(i['w']>0)) for i in val)
    df['inner_role']='excluded_guard_or_shared_exhaustive_unit'
    df.loc[val_df.index,'inner_role']='inner_val_candidate'
    df.loc[train_df.index,'inner_role']='inner_train'
    df.to_csv(OUT/f'03_v4_002_inner_spatial_split_fold{fold}.csv',index=False)
    print('INNER SPATIAL SPLIT',fold,'training',len(train),
          'selection_exhaustive_only',len(val),
          'excluded',int(df.inner_role.eq('excluded_guard_or_shared_exhaustive_unit').sum()))
    return train,val

def save_source_audit(fold,train,val):
    rows=[]
    for role,items in [('inner_train',train),('inner_val',val)]:
        for item in items:
            src=item['source']
            y=item['y'].astype(bool);w=item['w'].astype(bool)
            assert np.array_equal(src>0,w),('Provenance inconsistent',fold,role)
            if role=='inner_val':assert not np.any(src==1),'Partial label leaked into inner validation'
            for code in (1,2,3):
                p=(src==code)&w;n=int(p.sum())
                if n:
                    rows.append(dict(fold=fold,role=role,site=item['site'],
                        mosaic=item['mosaic'],source=SOURCE_CODES[code],
                        labelled_pixels=n,positive_pixels=int((p&y).sum()),
                        negative_pixels=int((p&~y).sum()),
                        chip_key=f"{item['mosaic']}|{item['row']}|{item['col']}"))
    summary=pd.DataFrame(rows).groupby(
        ['fold','role','site','source'],as_index=False).agg(
        n_chips=('chip_key','nunique'),
        labelled_pixels=('labelled_pixels','sum'),
        positive_pixels=('positive_pixels','sum'),
        negative_pixels=('negative_pixels','sum'))
    assert (summary.positive_pixels+summary.negative_pixels==summary.labelled_pixels).all()
    assert not summary.loc[summary.role.eq('inner_val'),'source'].eq('partial_crown').any()
    summary.to_csv(OUT/f'03_v4_002_annotation_source_audit_fold{fold}.csv',index=False)
    print('SOURCE AUDIT fold',fold,'(partial training ONLY):')
    display(summary)


In [ ]:
# 05 — Restore the working V3_1 ResNet18 U-Net + masked BCE, AMP in training
import segmentation_models_pytorch as smp
import torch.nn.functional as F
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
            in_channels=6,classes=1)
    def forward(self,x):return self.net(x)  # B,1,H,W
def masked_loss(logits,y,weight):
    """Original V3_1 masked pixelwise BCE; unknown/unlabelled pixels have weight=0."""
    bce=F.binary_cross_entropy_with_logits(logits.float(),y.float(),reduction='none')
    return (bce*weight).sum()/weight.sum().clamp_min(1.)

AMP=DEVICE.type=='cuda'
def run_epoch(model,loader,optimizer=None,scaler=None):
    model.train(optimizer is not None)
    totals=[];site_losses={}
    for x,y,w,site,src in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        w=w.to(DEVICE,non_blocking=True)
        src=src.to(DEVICE,non_blocking=True)
        if not w.any():continue
        with torch.set_grad_enabled(optimizer is not None):
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                logits=model(x)
            if optimizer is None:
                assert not torch.any((src==1)&(w>0)), 'PARTIAL CROWNS IN MODEL SELECTION'
                assert not torch.any((src==0)&(w>0)), 'UNKNOWN PIXELS IN MODEL SELECTION'
            loss=masked_loss(logits,y,w)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer);scaler.update()
        totals.append(loss.detach().item())
        if optimizer is None:
            # Per-chip, equal-site loss (avoid ForestGEO pixel-volume dominance)
            for i,name in enumerate(site):
                if w[i].sum()==0:continue
                l=masked_loss(logits[i:i+1],y[i:i+1],w[i:i+1])
                site_losses.setdefault(str(name),[]).append(float(l.item()))
    if optimizer is None:
        return float(np.mean([np.mean(v) for v in site_losses.values()])) if site_losses else float('inf')
    return float(np.mean(totals)) if totals else float('inf')


In [ ]:
# 04 — V3_1-style RAM-backed dataset (no GDAL calls inside GPU training)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class CachedChips(Dataset):
    def __init__(self,items,augment=False):
        self.items=items;self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'];y=q['y'];w=q['w'];src=q['source']
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            rgb=np.rot90(rgb,k);y=np.rot90(y,k);w=np.rot90(w,k);src=np.rot90(src,k)
            if torch.rand(1).item()<.5:
                rgb=rgb[:,::-1];y=y[:,::-1];w=w[:,::-1];src=src[:,::-1]
        x=torch.from_numpy(np.ascontiguousarray(rgb.transpose(2,0,1))).float()/255.
        return x,torch.from_numpy(np.ascontiguousarray(y[None])).float(),torch.from_numpy(np.ascontiguousarray(w[None])).float(),q['site'],torch.from_numpy(np.ascontiguousarray(src[None])).long()
def cached_sampler(items):
    counts=pd.Series([r['site'] for r in items]).value_counts().to_dict()
    weights=[(POSITIVE_SAMPLE_BOOST if np.any(r['y']&r['w']) else 1.)/counts[r['site']]
             for r in items]
    return WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
        num_samples=min(len(items),STEPS_PER_EPOCH*BATCH_SIZE),replacement=True)
print('Weighted sampler for DSM chips ready.')


In [ ]:
# 04 — DSM-backed model cache + fold-safe normalisation
from contextlib import ExitStack
import gc, time
from torch.utils.data import Dataset,DataLoader
import torch
from sklearn.metrics import r2_score
from rasterio.windows import Window
from rasterio.features import rasterize
def add_dsm_to_items(items,fold):
    out=[]
    for ix,item in enumerate(items):
        name=item['mosaic']
        assert name in REGISTRY,(fold,name)
        win=Window(item['col'],item['row'],CHIP,CHIP)
        try:
            v,joint=feature_tile(REGISTRY[name]['rgb'],REGISTRY[name]['dsm'],win)
        except Exception as exc:
            raise RuntimeError(f'DSM failure fold {fold} chip {ix} {name} {win}') from exc
        q=dict(item)
        q['dsm']=v[3:].astype(np.float16)
        q['joint']=joint.astype(np.uint8)
        q['w']=(q['w'].astype(bool)&joint).astype(np.uint8)
        q['source']=np.where(q['w']>0,q['source'],0).astype(np.uint8)
        # Reject only chips with insufficient matched annotation support
        if q['w'].sum()>=50:out.append(q)
        if (ix+1)%50==0:print('DSM chips',fold,ix+1,'/',len(items),flush=True)
    assert out,('No matched DSM chips in fold',fold)
    print('Fold',fold,'matched chips',len(out),'/',len(items))
    return out

def norm_from_training(items):
    # STRICTLY training pixels from the inner training partition.
    s=np.zeros(3,dtype=np.float64);ss=np.zeros(3,dtype=np.float64);count=0
    for q in items:
        support=q['joint'].astype(bool)&q['w'].astype(bool)
        if not support.any():continue
        a=q['dsm'].astype(np.float32)[:,support].astype(np.float64)
        s+=a.sum(axis=1);ss+=(a*a).sum(axis=1);count+=a.shape[1]
    assert count>1000,'Insufficient DSM training pixels'
    mu=s/count;sd=np.sqrt(np.maximum(ss/count-mu*mu,1.e-8))
    assert np.isfinite(mu).all() and (sd>0).all()
    return mu.astype(np.float32),sd.astype(np.float32)

class DSMChips(Dataset):
    def __init__(self,items,mu,sd,augment=False):
        self.items=items;self.mu=mu[:,None,None];self.sd=sd[:,None,None]
        self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=np.asarray(q['rgb'],np.float32).transpose(2,0,1)/255.
        dsm=np.clip((q['dsm'].astype(np.float32)-self.mu)/self.sd,-6,6)
        x=np.concatenate((rgb,dsm),axis=0)
        y=q['y'][None].astype(np.float32);w=q['w'][None].astype(np.float32)
        src=q['source'][None].astype(np.int64)
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k,axes=(1,2))
            w=np.rot90(w,k,axes=(1,2));src=np.rot90(src,k,axes=(1,2))
            if torch.rand(1).item()<.5:
                x=x[:,:,::-1];y=y[:,:,::-1];w=w[:,:,::-1];src=src[:,:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(w)),
                q['site'],torch.from_numpy(np.ascontiguousarray(src)))


## 04 — Fit RGB+DSM U-Net for three outer folds; skip existing verified folds

In [ ]:
# 05 — FULL 3-fold DSM training, with blocked inner checkpoint selection
# Stop on invalid DSM mapping/registration concerns. Preserve any completed checkpoints.
from torch.utils.data import WeightedRandomSampler
HISTORY=[]
for fold in RUN_FOLDS:
    ckpt=CHECKPOINTS[fold]
    if ckpt.exists():
        existing=torch.load(ckpt,map_location='cpu',weights_only=False)
        assert existing['version']=='V4.002' and existing['fold']==fold
        print('PRESERVE existing DSM checkpoint fold',fold)
        continue
    print('PREPARING DSM FOLD',fold,flush=True)
    items=make_cache(fold,'train')  # original V4 source-labelled RGB cache recipe
    items=add_dsm_to_items(items,fold)
    inner_train,inner_val=inner_spatial_split(items,fold)
    save_source_audit(fold,inner_train,inner_val)
    mu,sd=norm_from_training(inner_train)
    print('DSM training-only mean/sd:',mu,sd)
    trainloader=DataLoader(DSMChips(inner_train,mu,sd,True),
        batch_size=BATCH_SIZE,sampler=cached_sampler(inner_train),
        num_workers=0,pin_memory=AMP)
    valloader=DataLoader(DSMChips(inner_val,mu,sd,False),
        batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=AMP)
    model=ResNet18UNet().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    best=float('inf');wait=0
    for epoch in range(1,EPOCHS+1):
        tr=run_epoch(model,trainloader,opt,scaler)
        vl=run_epoch(model,valloader)
        H=dict(fold=fold,epoch=epoch,train_loss=tr,val_loss=vl,
               n_train=len(inner_train),n_val=len(inner_val))
        HISTORY.append(H);print(H,flush=True)
        if vl<best-1e-4:
            best=vl;wait=0
            torch.save(dict(model=model.state_dict(),fold=fold,version='V4.002',
                architecture='smp_unet_resnet18_imagenet_6channel',
                model_inputs=list(FEATURE_NAMES),norm_mean=mu,norm_std=sd,
                epoch=epoch,inner_val_loss=best,
                n_train_chips=len(inner_train),n_val_chips=len(inner_val),
                selection_protocol='inner_exhaustive_only_blocked_plus_unit_guard',
                source_loss='V3_1_masked_pixel_BCE',
                feature_windows_m=[PROMINENCE_RADIUS_M,ROUGHNESS_RADIUS_M,CONVEXITY_RADIUS_M],
                rgb_baseline_checkpoint=str(BASELINE_CHECKPOINTS[fold])),ckpt)
        else:
            wait+=1
            if wait>=PATIENCE:break
    pd.DataFrame(HISTORY).to_csv(OUT/'03_training_history.csv',index=False)
    del items,inner_train,inner_val,trainloader,valloader,model
    torch.cuda.empty_cache();gc.collect()
assert all(x.exists() for x in CHECKPOINTS.values())
print('ALL THREE DSM FOLDS TRAINED AND FROZEN')


## 05 — Canonical V4.009 probability and metric helpers; matched inference

In [ ]:
# GLOBAL 128 px grid, 256 px tiles, Hann blend — model probabilities, not hard masks.
# Every output core is evaluated ONCE; the outer halo supplies inference context.
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from pyproj import Transformer
import gc

INFERENCE_VERSION='V4.009'
EVAL_OUT=OUT/'paired_evaluation'
EVAL_OUT.mkdir(parents=True,exist_ok=True)
V009_TILE=256
V009_STRIDE=128
assert VERSION=='V4.007' and INFERENCE_VERSION=='V4.009'
assert V009_TILE==CHIP and V009_STRIDE==CHIP//2
for fold,p in BASELINE_CHECKPOINTS.items():
    assert p.is_file(),('Missing frozen V4.007 RGB checkpoint',fold,p)
FOLDS=[0,1,2]
PROBABILITY_BINS=512
PLOT_PROB_HISTS={}
CIRCLE_PROB_HISTS={}
EXAMPLE_IDS={3,9,29,26}
SAVE_EXAMPLE_FIGURES=True

def load_frozen_model(fold):
    ck=torch.load(CHECKPOINTS[int(fold)],map_location='cpu',weights_only=False)
    assert ck['version']=='V4.007' and ck['fold']==int(fold) and not ck['smoke']
    assert ck.get('source_loss')=='V3_1_masked_pixel_BCE'
    m=ResNet18UNet().to(DEVICE)
    m.load_state_dict(ck['model'])
    del ck
    m.eval()
    return m

def burn(gs,ds,win):
    if not gs:return np.zeros((int(win.height),int(win.width)),dtype=bool)
    return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
        out_shape=(int(win.height),int(win.width)),
        transform=ds.window_transform(win),fill=0,dtype='uint8').astype(bool)

# Deliberately nonzero edge weight to guarantee complete support at raster limits.
hann=np.hanning(V009_TILE+2)[1:-1].astype(np.float32)
WEIGHT=np.maximum(np.outer(hann,hann),0.01).astype(np.float32)
WEIGHT/=WEIGHT.max()

def blended_window(ds,model,win):
    """Blended P(Musanga) for a core window, anchored to entire mosaic pixel grid.

    Loads only bounded core+halo RGB. No per-tile compressed-cache calls, no
    complete-orthomosaic arrays, no thresholding before probability blending.
    """
    core_r=int(win.row_off);core_c=int(win.col_off)
    h=int(win.height);w=int(win.width)
    assert h>0 and w>0 and core_r>=0 and core_c>=0
    assert core_r+h<=ds.height and core_c+w<=ds.width
    halo=V009_TILE
    top=max(0,core_r-halo);left=max(0,core_c-halo)
    bottom=min(ds.height,core_r+h+halo);right=min(ds.width,core_c+w+halo)
    full=Window(left,top,right-left,bottom-top)
    raw=ds.read([1,2,3],window=full)
    assert ds.dtypes[0]=='uint8',(ds.name,ds.dtypes)
    rgb=raw.astype(np.float32)/255.
    H,W=rgb.shape[1:]
    acc=np.zeros((H,W),np.float32)
    wt=np.zeros((H,W),np.float32)
    # Every contributing patch aligned to global origin at steps of 128 px.
    r0=((top-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    c0=((left-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    with torch.inference_mode():
        for gr in range(r0,bottom,V009_STRIDE):
            for gc0 in range(c0,right,V009_STRIDE):
                ar=max(top,gr);br=min(bottom,gr+V009_TILE)
                ac=max(left,gc0);bc=min(right,gc0+V009_TILE)
                if br<=ar or bc<=ac:continue
                patch=np.zeros((3,V009_TILE,V009_TILE),np.float32)
                patch[:,ar-gr:br-gr,ac-gc0:bc-gc0]=rgb[:,ar-top:br-top,ac-left:bc-left]
                xx=torch.from_numpy(patch[None]).to(DEVICE)
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                    p=torch.sigmoid(model(xx))[0,0].float().cpu().numpy()
                z=WEIGHT[ar-gr:br-gr,ac-gc0:bc-gc0]
                acc[ar-top:br-top,ac-left:bc-left]+=p[ar-gr:br-gr,ac-gc0:bc-gc0]*z
                wt[ar-top:br-top,ac-left:bc-left]+=z
    oy=core_r-top;ox=core_c-left
    sl=np.s_[oy:oy+h,ox:ox+w]
    assert np.all(wt[sl]>0),(ds.name,win,'uncovered prediction pixels')
    return (acc[sl]/wt[sl]).astype(np.float32)

def collect_metrics(p,truth,support):
    assert support.any()
    pred=p>=0.5
    tp=int((pred&truth&support).sum());fp=int((pred&~truth&support).sum())
    fn=int((~pred&truth&support).sum());tn=int((~pred&~truth&support).sum())
    n=int(support.sum())
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,n=n,
        observed_cover_pct=100*(tp+fn)/n,
        predicted_cover_pct=100*(tp+fp)/n,
        predicted_probability_pct=100*float(p[support].mean()),
        bias_pp=100*(fp-fn)/n,
        Brier=float(np.mean((p[support]-truth[support])**2)),
        **confusion_statistics(tp,fp,fn,tn))

def _figure(cid,mosaic,ds,win,rgb,truth,support,prob,scores):
    ds_h,ds_w=truth.shape
    step=max(1,int(np.ceil(max(ds_h,ds_w)/950)))
    sl=np.s_[::step,::step]
    view=rgb.transpose(1,2,0)[sl]
    y=truth[sl];supp=support[sl];p=prob[sl]
    cls=p>=0.5
    fig,axs=plt.subplots(1,5,figsize=(20,4.7),layout='constrained')
    axs[0].imshow(view);axs[0].set_title('RGB')
    overlay=view.copy();overlay[y&supp]=.45*overlay[y&supp]+.55*np.array([1,.1,.1])
    axs[1].imshow(np.clip(overlay,0,1));axs[1].set_title('Exhaustive reference · red')
    axs[2].imshow(view)
    a=axs[2].imshow(np.ma.masked_where(~supp,p),cmap='magma',vmin=0,vmax=1,alpha=.90)
    axs[2].set_title('Blended probability P(Musanga)')
    fig.colorbar(a,ax=axs[2],fraction=.045,pad=.02)
    overlay=view.copy();overlay[cls&supp]=.45*overlay[cls&supp]+.55*np.array([.1,1,1])
    axs[3].imshow(np.clip(overlay,0,1));axs[3].set_title('Blended threshold 0.5 · cyan')
    err=view.copy()
    for mask,color in [(cls&y&supp,[.1,1,.1]),(cls&~y&supp,[1,.1,.1]),(~cls&y&supp,[.1,.35,1])]:
        err[mask]=.3*err[mask]+.7*np.array(color)
    axs[4].imshow(np.clip(err,0,1));axs[4].set_title('TP green · FP red · FN blue')
    for ax in axs:ax.axis('off')
    fig.suptitle(f'{mosaic} | circle {cid} | observed {scores["observed_cover_pct"]:.1f}% '
                 f'→ blended {scores["predicted_cover_pct"]:.1f}% | '
                 f'bias {scores["bias_pp"]:+.1f} pp',fontsize=12)
    fig.savefig(EVAL_OUT/f'circle_{cid:02d}_blended_probability.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)
def coverage_summary(g):
    """Site-pooled cover metrics and mean per-circle/plot pixel statistics."""
    yy=g.observed_cover_pct.to_numpy(float)
    pp=g.predicted_cover_pct.to_numpy(float)
    return pd.Series(dict(n=len(g),
        R2=r2_score(yy,pp) if len(g)>1 else np.nan,
        MAE_pp=np.mean(np.abs(pp-yy)),
        RMSE_pp=np.sqrt(np.mean((pp-yy)**2)),
        bias_pp=np.mean(pp-yy),
        mean_pixel_F1=g.pixel_F1.mean(),
        mean_pixel_IoU=g.pixel_IoU.mean(),
        mean_Brier=g.Brier.mean()))


def probability_histogram(probabilities,truth):
    """512-bin streaming histogram; preserves all eligible pixels but no huge arrays."""
    p=np.asarray(probabilities,dtype=np.float32).ravel()
    y=np.asarray(truth,dtype=bool).ravel()
    assert len(p)==len(y) and np.isfinite(p).all()
    b=np.minimum((np.clip(p,0.,1.)*PROBABILITY_BINS).astype(np.int32),
                 PROBABILITY_BINS-1)
    total=np.bincount(b,minlength=PROBABILITY_BINS).astype(np.int64)
    positive=np.bincount(b,weights=y.astype(np.int64),
                         minlength=PROBABILITY_BINS).astype(np.int64)
    assert positive.sum()==int(y.sum())
    return total,positive

def rank_scores(total,positive):
    """Approximate ROC-AUC / AP using 512 probability bins, with ties accounted for."""
    total=np.asarray(total,dtype=np.float64)
    pos=np.asarray(positive,dtype=np.float64)
    neg=total-pos
    npos=float(pos.sum());nneg=float(neg.sum())
    if npos==0 or nneg==0:return np.nan,np.nan
    auc=float(np.dot(pos,np.cumsum(neg)-.5*neg)/(npos*nneg))
    p_desc=pos[::-1];n_desc=neg[::-1]
    recall_step=p_desc/npos
    precision=np.cumsum(p_desc)/np.maximum(1.,np.cumsum(p_desc+n_desc))
    ap=float(np.dot(recall_step,precision))
    return auc,ap

def confusion_statistics(tp,fp,fn,tn):
    tp,fp,fn,tn=[int(z) for z in (tp,fp,fn,tn)]
    pos=tp+fn;neg=tn+fp;predpos=tp+fp;den=tp+fp+fn
    prec=tp/predpos if predpos else np.nan
    rec=tp/pos if pos else np.nan
    spec=tn/neg if neg else np.nan
    return dict(pixel_precision=prec,pixel_recall=rec,
        pixel_specificity=spec,
        pixel_balanced_accuracy=(rec+spec)/2 if np.isfinite(rec) and np.isfinite(spec) else np.nan,
        pixel_F1=2*tp/(2*tp+fp+fn) if pos and (2*tp+fp+fn) else np.nan,
        pixel_IoU=tp/den if pos and den else np.nan,
        pixel_prevalence=pos/(pos+neg) if pos+neg else np.nan)

def complete_cover_summary(g,hists=None,unit_key=None):
    """Macro cover skill, macro pixel stats, pooled pixel stats, binned probability scores."""
    yy=g.observed_cover_pct.to_numpy(float)
    pp=g.predicted_cover_pct.to_numpy(float)
    valid=np.isfinite(yy)&np.isfinite(pp)
    yy=yy[valid];pp=pp[valid]
    assert len(yy)>0
    diff=pp-yy
    rmse=float(np.sqrt(np.mean(diff**2)))
    sd=float(np.std(yy,ddof=1)) if len(yy)>1 else np.nan
    iqr=float(np.percentile(yy,75)-np.percentile(yy,25)) if len(yy)>1 else np.nan
    r=float(np.corrcoef(yy,pp)[0,1]) if len(yy)>1 and np.std(yy)>1e-10 and np.std(pp)>1e-10 else np.nan
    slope=float(np.cov(yy,pp,ddof=0)[0,1]/np.var(yy)) if len(yy)>1 and np.var(yy)>1e-15 else np.nan
    intercept=float(np.mean(pp)-slope*np.mean(yy)) if np.isfinite(slope) else np.nan
    tp=int(g.tp.sum());fp=int(g.fp.sum());fn=int(g.fn.sum());tn=int(g.tn.sum())
    pooled=confusion_statistics(tp,fp,fn,tn)
    result=dict(n_units=len(yy),
        cover_R2=float(r2_score(yy,pp)) if len(yy)>1 and np.var(yy)>1e-15 else np.nan,
        cover_Pearson_r=r,cover_MAE_pp=float(np.mean(abs(diff))),
        cover_RMSE_pp=rmse,cover_bias_pp=float(np.mean(diff)),
        cover_RPD=sd/rmse if rmse>0 else np.nan,
        cover_RPIQ=iqr/rmse if rmse>0 else np.nan,
        calibration_slope=slope,calibration_intercept_pp=intercept,
        macro_pixel_F1=float(g.pixel_F1.mean()),macro_pixel_IoU=float(g.pixel_IoU.mean()),
        macro_pixel_precision=float(g.pixel_precision.mean()),
        macro_pixel_recall=float(g.pixel_recall.mean()),
        macro_pixel_specificity=float(g.pixel_specificity.mean()),
        macro_pixel_balanced_accuracy=float(g.pixel_balanced_accuracy.mean()),
        mean_Brier=float(g.Brier.mean()),
        pooled_Brier=float(np.average(g.Brier,weights=g.n)) if int(g.n.sum()) else np.nan,
        pooled_tp=tp,pooled_fp=fp,pooled_fn=fn,pooled_tn=tn)
    result.update({'pooled_'+k:v for k,v in pooled.items()})
    if hists is not None:
        assert unit_key is not None
        selected=[hists[key] for key in g[unit_key] if key in hists]
        if selected:
            total=np.sum([x[0] for x in selected],axis=0)
            positive=np.sum([x[1] for x in selected],axis=0)
            assert int(total.sum())==int(g.n.sum()),'Histogram and support totals disagree'
            auc,ap=rank_scores(total,positive)
        else:
            auc,ap=np.nan,np.nan
        result.update(pixel_ROC_AUC_512bins=auc,pixel_average_precision_512bins=ap)
    return pd.Series(result)

def metrics_table(df,kind,hists,key):
    eligible=df.loc[df.n.gt(0)].copy()
    summaries=[dict(grouping='ALL',site='ALL',fold='ALL',
                    **complete_cover_summary(eligible,hists,key).to_dict())]
    for site,g in eligible.groupby('site'):
        summaries.append(dict(grouping='site',site=site,fold='ALL',
                     **complete_cover_summary(g,hists,key).to_dict()))
    for fold,g in eligible.groupby('fold'):
        summaries.append(dict(grouping='fold',site='ALL',fold=int(fold),
                     **complete_cover_summary(g,hists,key).to_dict()))
    for (fold,site),g in eligible.groupby(['fold','site']):
        summaries.append(dict(grouping='site_fold',site=site,fold=int(fold),
                     **complete_cover_summary(g,hists,key).to_dict()))
    out=pd.DataFrame(summaries)
    out.to_csv(EVAL_OUT/f'V4_009_{kind}_complete_metric_report.csv',index=False)
    return out

def save_histograms(path,hists):
    arrays={}
    for key,(tot,pos) in hists.items():
        safe=str(key).replace('/','_').replace(' ','_')
        arrays[safe+'_total']=tot
        arrays[safe+'_positive']=pos
    np.savez_compressed(path,**arrays)

print('READY — V4.009 unified 50% overlap + smooth blend, models read-only')


In [ ]:
# 06 — Paired RGB / DSM global-grid 50%-overlap probabilities
# Both models receive identical RGB patches and the same tile origin / Hann weights.
def paired_blended(ds,model_rgb,model_dsm,win,mu,sd,dsm_path):
    rr=int(win.row_off);cc=int(win.col_off);h=int(win.height);w=int(win.width)
    halo=V009_TILE
    top=max(0,rr-halo);left=max(0,cc-halo)
    bottom=min(ds.height,rr+h+halo);right=min(ds.width,cc+w+halo)
    full=Window(left,top,right-left,bottom-top)
    all_features,joint=feature_tile(ds.name,dsm_path,full)
    rgb=ds.read([1,2,3],window=full).astype(np.float32)/255.
    features=all_features[3:]
    features=np.clip((features-mu[:,None,None])/sd[:,None,None],-6,6)
    H,W=joint.shape
    acc0=np.zeros((H,W),np.float32);acc1=np.zeros((H,W),np.float32)
    wt=np.zeros((H,W),np.float32)
    r0=((top-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    c0=((left-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    with torch.inference_mode():
        for gr in range(r0,bottom,V009_STRIDE):
            for gc0 in range(c0,right,V009_STRIDE):
                ar=max(top,gr);br=min(bottom,gr+V009_TILE)
                ac=max(left,gc0);bc=min(right,gc0+V009_TILE)
                if br<=ar or bc<=ac:continue
                xx=np.zeros((6,V009_TILE,V009_TILE),np.float32)
                sl_src=np.s_[ar-top:br-top,ac-left:bc-left]
                sl_dst=np.s_[ar-gr:br-gr,ac-gc0:bc-gc0]
                xx[:3,sl_dst[0],sl_dst[1]]=rgb[:,sl_src[0],sl_src[1]]
                xx[3:,sl_dst[0],sl_dst[1]]=features[:,sl_src[0],sl_src[1]]
                t=torch.from_numpy(xx[None]).to(DEVICE)
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                    p0=torch.sigmoid(model_rgb(t[:,:3]))[0,0].float().cpu().numpy()
                    p1=torch.sigmoid(model_dsm(t))[0,0].float().cpu().numpy()
                z=WEIGHT[sl_dst]
                acc0[sl_src]+=p0[sl_dst]*z
                acc1[sl_src]+=p1[sl_dst]*z
                wt[sl_src]+=z
    sy=rr-top;sx=cc-left;core=np.s_[sy:sy+h,sx:sx+w]
    assert np.all(wt[core]>0)
    return acc0[core]/wt[core],acc1[core]/wt[core],joint[core]

def load_pair(fold):
    rgb=load_frozen_model_rgb(fold)
    ck=torch.load(CHECKPOINTS[fold],map_location='cpu',weights_only=False)
    assert ck['version']=='V4.002' and ck['fold']==fold
    dsm=ResNet18UNet().to(DEVICE)
    dsm.load_state_dict(ck['model']);dsm.eval()
    return rgb,dsm,np.asarray(ck['norm_mean'],np.float32),np.asarray(ck['norm_std'],np.float32)

def load_frozen_model_rgb(fold):
    ck=torch.load(BASELINE_CHECKPOINTS[fold],map_location='cpu',weights_only=False)
    assert ck['version']=='V4.007' and ck['fold']==fold
    m=smp.Unet(encoder_name='resnet18',encoder_weights=None,
               in_channels=3,classes=1).to(DEVICE)
    # V4.007 checkpoint stores outer module wrapper with a 'net.' prefix.
    m.load_state_dict({k[4:]:v for k,v in ck['model'].items()})
    m.eval()
    return m


## 06 — Paired held-out field plots and circles, full report

In [ ]:
# 07 — Both models, identical supports and spatial folds; plots + circles
from collections import defaultdict
def unit_accum():
    return dict(tp=0,fp=0,fn=0,tn=0,n=0,
        total=np.zeros(PROBABILITY_BINS,np.int64),
        positive=np.zeros(PROBABILITY_BINS,np.int64))
def update_unit(u,p,y):
    if not len(p):return
    q=p>=.5
    u['tp']+=int((q&y).sum());u['fp']+=int((q&~y).sum())
    u['fn']+=int((~q&y).sum());u['tn']+=int((~q&~y).sum())
    u['n']+=len(y)
    a,b=probability_histogram(p,y)
    u['total']+=a;u['positive']+=b
def finish_unit(uid,site,fold,variant,kind,u):
    n=u['n'];tp=u['tp'];fp=u['fp'];fn=u['fn'];tn=u['tn']
    if n==0:return None
    d=dict(unit_id=uid,site=site,fold=fold,variant=variant,kind=kind,n=n,
        observed_cover_pct=100*(tp+fn)/n,
        predicted_cover_pct=100*(tp+fp)/n,
        cover_error_pp=100*(fp-fn)/n,
        **{k:u[k] for k in ('tp','fp','fn','tn')},
        **confusion_statistics(tp,fp,fn,tn))
    auc,ap=rank_scores(u['total'],u['positive'])
    d.update(ROC_AUC=auc,average_precision=ap)
    return d
RESULTS=[]
for fold in FOLDS:
    model0,model1,mu,sd=load_pair(fold)
    print('PAIRED EVALUATION FOLD',fold,flush=True)
    plot_acc={(str(q.Plot_name),variant):unit_accum()
              for q in PLOTS.loc[PLOTS.cv_fold.eq(fold)].itertuples(index=False)
              for variant in ('RGB','RGB_DSM')}
    rows=SPLITS[(SPLITS.cv_fold==fold)&SPLITS.split.eq('val')]
    for mosaic,g in rows.groupby('mosaic',sort=True):
        with rasterio.open(RGB[mosaic]['path']) as ds:
            for z in g.itertuples(index=False):
                win=Window(int(z.col),int(z.row),int(z.width),int(z.height))
                bb=box(*rasterio.windows.bounds(win,ds.transform))
                hits=[PLOTS.iloc[int(i)] for i in P_T.query(bb,predicate='intersects')
                    if PLOTS.iloc[int(i)].site_key==z.site and
                       int(PLOTS.iloc[int(i)].cv_fold)==fold]
                if not hits:continue
                roi=burn(geom_hits(ROI_T,ROI_GEOMS,bb),ds,win)
                pos=burn(geom_hits(CP_T,CROWN_POS,bb),ds,win)
                p0,p1,joint=paired_blended(ds,model0,model1,win,mu,sd,REGISTRY[mosaic]['dsm'])
                for pl in hits:
                    support=burn([pl.geometry],ds,win)&roi&joint
                    if not support.any():continue
                    y=pos[support]
                    update_unit(plot_acc[str(pl.Plot_name),'RGB'],p0[support],y)
                    update_unit(plot_acc[str(pl.Plot_name),'RGB_DSM'],p1[support],y)
    for pl in PLOTS.loc[PLOTS.cv_fold.eq(fold)].itertuples(index=False):
        for variant in ('RGB','RGB_DSM'):
            record=finish_unit(str(pl.Plot_name),pl.site_key,fold,variant,'plot',
                 plot_acc[str(pl.Plot_name),variant])
            if record:RESULTS.append(record)
    print('PLOT FOLD FINISHED',fold,flush=True)
    # All 34 circles; no reinterpretation of circle labels as field plots.
    for cr in CIRCLE_ROI.loc[CIRCLE_ROI.cv_fold.eq(fold)].itertuples(index=False):
        matches=[]
        for name,meta in RGB.items():
            if meta['site']!=cr.site_key:continue
            with rasterio.open(meta['path']) as ds:
                matches.append((cr.geometry.intersection(box(*ds.bounds)).area,name))
        area,mosaic=max(matches)
        assert area>0,(cr.circle_id,'No RGB overlap')
        with rasterio.open(RGB[mosaic]['path']) as ds:
            bounds=rasterio.windows.from_bounds(*cr.geometry.bounds,transform=ds.transform)
            r=max(0,int(np.floor(bounds.row_off)))
            c=max(0,int(np.floor(bounds.col_off)))
            bottom=min(ds.height,int(np.ceil(bounds.row_off+bounds.height)))
            right=min(ds.width,int(np.ceil(bounds.col_off+bounds.width)))
            win=Window(c,r,right-c,bottom-r)
            bb=box(*rasterio.windows.bounds(win,ds.transform))
            truth=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,bb),ds,win)
            mask=burn([cr.geometry],ds,win)
            roi=burn(geom_hits(ROI_T,ROI_GEOMS,bb),ds,win)
            p0,p1,joint=paired_blended(ds,model0,model1,win,mu,sd,REGISTRY[mosaic]['dsm'])
            support=mask&roi&joint
            for variant,p in [('RGB',p0),('RGB_DSM',p1)]:
                u=unit_accum();update_unit(u,p[support],truth[support])
                rec=finish_unit(int(cr.circle_id),cr.site_key,fold,variant,'circle',u)
                if rec:RESULTS.append(rec)
    del model0,model1
    torch.cuda.empty_cache();gc.collect()
    pd.DataFrame(RESULTS).to_csv(EVAL_OUT/'03_paired_units_partial.csv',index=False)
    print('FOLD DONE',fold,flush=True)
R=pd.DataFrame(RESULTS)
R.to_csv(EVAL_OUT/'03_paired_units.csv',index=False)
assert all(R[R.kind.eq(kind)].groupby('variant').unit_id.nunique().eq(
    R[R.kind.eq(kind)].unit_id.nunique()) for kind in ('plot','circle'))
rows=[]
for (kind,site,variant),g in R.groupby(['kind','site','variant']):
    obs=g.observed_cover_pct.to_numpy();pred=g.predicted_cover_pct.to_numpy()
    rows.append(dict(kind=kind,site=site,variant=variant,n=len(g),
      cover_R2=r2_score(obs,pred) if len(g)>1 and np.var(obs)>1e-12 else np.nan,
      MAE_pp=float(np.mean(abs(obs-pred))),RMSE_pp=float(np.sqrt(np.mean((obs-pred)**2))),
      bias_pp=float(np.mean(pred-obs)),
      pooled_F1=2*g.tp.sum()/max(1,2*g.tp.sum()+g.fp.sum()+g.fn.sum()),
      pooled_IoU=g.tp.sum()/max(1,g.tp.sum()+g.fp.sum()+g.fn.sum())))
for (kind,variant),g in R.groupby(['kind','variant']):
    obs=g.observed_cover_pct.to_numpy();pred=g.predicted_cover_pct.to_numpy()
    rows.append(dict(kind=kind,site='ALL',variant=variant,n=len(g),
      cover_R2=r2_score(obs,pred) if len(g)>1 and np.var(obs)>1e-12 else np.nan,
      MAE_pp=float(np.mean(abs(obs-pred))),RMSE_pp=float(np.sqrt(np.mean((obs-pred)**2))),
      bias_pp=float(np.mean(pred-obs)),
      pooled_F1=2*g.tp.sum()/max(1,2*g.tp.sum()+g.fp.sum()+g.fn.sum()),
      pooled_IoU=g.tp.sum()/max(1,g.tp.sum()+g.fp.sum()+g.fn.sum())))
SUMMARY=pd.DataFrame(rows).sort_values(['kind','site','variant'])
SUMMARY.to_csv(EVAL_OUT/'03_paired_summary.csv',index=False)
display(SUMMARY.round(3))
print('COMPLETE: matched RGB vs DSM, field plots and 34 circles')
